# L2 TinyBERT Sequence Classifier

Fine-tunes TinyBERT (14M params) on L1 out-of-fold residuals for
prompt injection detection. Exports to ONNX for Rust inference.

**Steps:**
1. Upload `l2b_training_candidates.jsonl` (from parapet-runner/runs/l1b_residuals_v8/)
2. Run all cells
3. Download `tinybert_l2/` (ONNX model + tokenizer)

Runtime: ~5-10 min on Colab free tier (T4 GPU).

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets onnx onnxruntime onnxscript scikit-learn

In [ ]:
# Cell 2: Upload residuals file
from google.colab import files
print("Upload l2b_training_candidates.jsonl:")
uploaded = files.upload()
RESIDUALS_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {RESIDUALS_FILE}")

In [ ]:
# Cell 3: Load and prepare data
import json
import random
from collections import Counter

residuals = []
with open(RESIDUALS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            residuals.append(json.loads(line))

print(f"Loaded {len(residuals)} residuals")

# Build training mix (same ratios as L1b experiments)
MIX = {"false_negative": 0.35, "false_positive": 0.25,
       "near_boundary_benign": 0.20, "baseline_correct": 0.20}

by_cat = {}
for r in residuals:
    cat = r.get("residual_category", "unknown")
    by_cat.setdefault(cat, []).append(r)

rng = random.Random(42)
fn_count = len(by_cat.get("false_negative", []))
total_target = int(fn_count / MIX["false_negative"])

mixed = []
for category, ratio in MIX.items():
    pool = by_cat.get(category, [])
    target = min(int(total_target * ratio), len(pool))
    sampled = rng.sample(pool, target) if target < len(pool) else list(pool)
    mixed.extend(sampled)
    print(f"  {category:<25} {len(sampled):>5}")

rng.shuffle(mixed)
print(f"\nTotal: {len(mixed)}")

labels = [1 if r["true_label"] == "malicious" else 0 for r in mixed]
texts = [r["content"] for r in mixed]
print(f"Malicious: {sum(labels)}, Benign: {len(labels) - sum(labels)}")

In [ ]:
# Cell 4: Train/val split
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)
print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")

In [ ]:
# Cell 5: Load model and tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 6: Create datasets
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }

train_dataset = TextDataset(train_texts, train_labels, tokenizer)
val_dataset = TextDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Cell 7: Train
from torch.optim import AdamW
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import numpy as np

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
EPOCHS = 5

best_val_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    # Train
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = total_loss / len(train_loader)

    # Validate
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in val_loader:
            batch_device = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch_device["input_ids"],
                attention_mask=batch_device["attention_mask"],
            )
            probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(batch["labels"].numpy())

    f1 = f1_score(all_labels, all_preds)
    p = precision_score(all_labels, all_preds)
    r = recall_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    print(f"Epoch {epoch+1}/{EPOCHS}: loss={avg_loss:.4f} "
          f"F1={f1:.4f} P={p:.4f} R={r:.4f} AUC={auc:.4f}")

    if f1 > best_val_f1:
        best_val_f1 = f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  -> New best!")

# Restore best
model.load_state_dict(best_state)
model = model.to(device)
print(f"\nBest val F1: {best_val_f1:.4f}")

In [ ]:
# Cell 8: Per-category and per-reason breakdown
from sklearn.model_selection import train_test_split as tts

# Reconstruct val indices using same seed/stratify as Cell 4
_, val_idx = tts(range(len(mixed)), test_size=0.15, random_state=42,
                 stratify=labels)
val_samples_full = [mixed[i] for i in val_idx]

model.eval()
all_preds_final = []
with torch.no_grad():
    for batch in val_loader:
        batch_device = {k: v.to(device) for k, v in batch.items()}
        outputs = model(
            input_ids=batch_device["input_ids"],
            attention_mask=batch_device["attention_mask"],
        )
        probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_preds_final.extend(preds)

print("Per-category accuracy:")
for cat in ["false_negative", "false_positive", "near_boundary_benign", "baseline_correct"]:
    mask = [i for i, s in enumerate(val_samples_full)
            if s.get("residual_category") == cat]
    if not mask:
        continue
    correct = sum(1 for i in mask if all_preds_final[i] == val_labels[i])
    print(f"  {cat:<25} {correct}/{len(mask)} ({correct/len(mask)*100:.1f}%)")

print("\nPer-reason recall (malicious only):")
reasons = sorted({s.get("reason") for s in val_samples_full
                  if s.get("true_label") == "malicious" and s.get("reason")})
for reason in reasons:
    idx = [i for i, s in enumerate(val_samples_full)
           if s.get("reason") == reason and s.get("true_label") == "malicious"]
    if not idx:
        continue
    correct = sum(1 for i in idx if all_preds_final[i] == 1)
    print(f"  {reason:<25} {correct}/{len(idx)} ({correct/len(idx)*100:.1f}%)")

print("\nPer-reason FPR (benign only):")
benign_reasons = sorted({s.get("reason") for s in val_samples_full
                         if s.get("true_label") == "benign" and s.get("reason")})
for reason in benign_reasons:
    idx = [i for i, s in enumerate(val_samples_full)
           if s.get("reason") == reason and s.get("true_label") == "benign"]
    if not idx:
        continue
    fp = sum(1 for i in idx if all_preds_final[i] == 1)
    print(f"  {reason:<25} {fp}/{len(idx)} ({fp/len(idx)*100:.1f}%)")

In [ ]:
# Cell 9: Export to ONNX
import os

OUTPUT_DIR = "tinybert_l2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.eval()
model_cpu = model.to("cpu")

# Dummy input with batch=2 so tracer doesn't hard-code batch=1
dummy = tokenizer(["test input one", "test input two"], return_tensors="pt",
                  padding=True, truncation=True, max_length=512)

# Force legacy TorchScript exporter (dynamo=True bakes static shapes despite dynamic_axes)
torch.onnx.export(
    model_cpu,
    (dummy["input_ids"], dummy["attention_mask"]),
    f"{OUTPUT_DIR}/model.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
        "logits": {0: "batch"},
    },
    opset_version=14,
    dynamo=False,
)

# Save tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

# Check file sizes
for f in os.listdir(OUTPUT_DIR):
    path = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  {f}: {size_mb:.1f} MB")

print(f"\nExported to {OUTPUT_DIR}/")

In [ ]:
# Cell 10: Verify ONNX inference matches PyTorch
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(f"{OUTPUT_DIR}/model.onnx")

# Test on a few samples
test_texts = val_texts[:10]
encoded = tokenizer(test_texts, return_tensors="np", padding=True,
                    truncation=True, max_length=512)

onnx_out = session.run(None, {
    "input_ids": encoded["input_ids"].astype(np.int64),
    "attention_mask": encoded["attention_mask"].astype(np.int64),
})

# Compare with PyTorch
with torch.no_grad():
    pt_encoded = tokenizer(test_texts, return_tensors="pt", padding=True,
                           truncation=True, max_length=512)
    pt_out = model_cpu(**pt_encoded).logits.numpy()

max_diff = np.abs(onnx_out[0] - pt_out).max()
print(f"ONNX vs PyTorch max diff: {max_diff:.2e}")
print("PASS" if max_diff < 1e-4 else "FAIL")

In [ ]:
# Cell 11: Download artifacts
# Zip the output directory
!zip -r tinybert_l2.zip tinybert_l2/

from google.colab import files
files.download("tinybert_l2.zip")
print("\nDownload started. Extract the zip into parapet/models/tinybert-l2/")